# 🧮 Neuronales Netz von Grund auf — nur mit NumPy

**Kein TensorFlow, kein PyTorch — nur Mathematik und NumPy!**

In diesem Notebook baust du ein komplettes neuronales Netz **von Grund auf**:

1. **Aktivierungsfunktionen**: ReLU, Sigmoid, Softmax
2. **Dense Layer**: Forward-Pass (y = xW + b) und Backpropagation
3. **Cross-Entropy Loss**: Mit numerisch stabilem Softmax
4. **SGD Optimizer**: Mit Momentum
5. **Komplettes Netzwerk**: Training auf MNIST

Jede Komponente wird einzeln erklärt und getestet — du verstehst **jede Zeile**!

In [ ]:
import numpy as np

print(f"NumPy Version: {np.__version__}")

---
## 1. Aktivierungsfunktionen

Aktivierungsfunktionen bringen **Nichtlinearität** ins Netzwerk. Ohne sie wäre das ganze Netz nur eine lineare Transformation!

### 1.1 ReLU (Rectified Linear Unit)

$$\text{ReLU}(x) = \max(0, x)$$

**Ableitung:**
$$\frac{d}{dx}\text{ReLU}(x) = \begin{cases} 1 & \text{wenn } x > 0 \\ 0 & \text{wenn } x \leq 0 \end{cases}$$

**Warum ReLU?**
- Einfach & schnell zu berechnen
- Kein "Vanishing Gradient"-Problem (wie bei Sigmoid)
- Führt zu spärlichen Aktivierungen (viele Nullen)

In [ ]:
class ReLU:
    """Rectified Linear Unit: f(x) = max(0, x)"""
    def __init__(self):
        self.cache = None  # Speichert Input für Backward-Pass

    def forward(self, x):
        self.cache = x
        return np.maximum(0, x)

    def backward(self, dout):
        # Gradient fließt nur durch, wo x > 0 war
        return dout * (self.cache > 0)


# Test
relu = ReLU()
x = np.array([[-1.0, 2.0], [3.0, -4.0]])
out = relu.forward(x)
print(f"Input:\n{x}")
print(f"ReLU Output:\n{out}")
print(f"\nBackward (dout=1):\n{relu.backward(np.ones_like(x))}")

### 1.2 Sigmoid

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

**Ableitung:**
$$\sigma'(x) = \sigma(x) \cdot (1 - \sigma(x))$$

**Eigenschaften:**
- Output immer zwischen 0 und 1
- Gut für binäre Klassifikation (letztes Layer)
- **Problem:** Vanishing Gradient für sehr große/kleine x

In [ ]:
class Sigmoid:
    """Sigmoid: f(x) = 1 / (1 + e^(-x))"""
    def __init__(self):
        self.cache = None

    def forward(self, x):
        # Clip verhindert Overflow bei exp()
        out = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        self.cache = out  # Speichert Output (nicht Input!)
        return out

    def backward(self, dout):
        # σ'(x) = σ(x) * (1 - σ(x))
        return dout * self.cache * (1 - self.cache)


# Test
sig = Sigmoid()
x = np.array([[0.0], [2.0], [-2.0]])
out = sig.forward(x)
print(f"Sigmoid(0)  = {out[0,0]:.4f}  (erwartet: 0.5)")
print(f"Sigmoid(2)  = {out[1,0]:.4f}  (erwartet: ~0.88)")
print(f"Sigmoid(-2) = {out[2,0]:.4f}  (erwartet: ~0.12)")

### 1.3 Softmax

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Wandelt Logits in **Wahrscheinlichkeiten** um (Summe = 1).

**Numerische Stabilität:** Wir subtrahieren das Maximum vor der Exponentialfunktion:
$$\text{Softmax}(x_i) = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

Das ändert das Ergebnis nicht (mathematisch äquivalent), verhindert aber Overflow.

In [ ]:
class Softmax:
    """Softmax mit numerischer Stabilität."""
    def __init__(self):
        self.cache = None

    def forward(self, x):
        shifted = x - np.max(x, axis=1, keepdims=True)  # Stabilität
        exp = np.exp(shifted)
        out = exp / np.sum(exp, axis=1, keepdims=True)
        self.cache = out
        return out

    def backward(self, dout):
        # Softmax + Cross-Entropy wird kombiniert (siehe Loss)
        return dout


# Test
sm = Softmax()
x = np.array([[1.0, 2.0, 3.0]])
probs = sm.forward(x)
print(f"Logits: {x}")
print(f"Softmax: {probs}")
print(f"Summe: {probs.sum():.4f}  (muss 1.0 sein) ✓")

---
## 2. Dense Layer (vollständig verbunden)

Der Grundbaustein jedes neuronalen Netzes:

**Forward:** $$y = x \cdot W + b$$

**Backward (Kettenregel):**
- $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot W^T$ (Gradient für vorherigen Layer)
- $\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial y}$ (Gradient für Gewichte)
- $\frac{\partial L}{\partial b} = \sum \frac{\partial L}{\partial y}$ (Gradient für Bias)

**He-Initialisierung:** $W \sim \mathcal{N}(0, \sqrt{2/n_{in}})$ — optimiert für ReLU

In [ ]:
class Dense:
    """Vollständig verbundener Layer: y = x @ W + b"""
    def __init__(self, input_dim, output_dim):
        # He-Initialisierung (gut für ReLU)
        self.W = np.random.randn(input_dim, output_dim) * np.sqrt(2.0 / input_dim)
        self.b = np.zeros((1, output_dim))
        self.cache = None  # (x, W) für Backward

    def forward(self, x):
        self.cache = (x, self.W)
        return x @ self.W + self.b

    def backward(self, dout):
        x, W = self.cache
        # Gradienten für W, b und Input
        self.dW = x.T @ dout          # (input_dim, output_dim)
        self.db = np.sum(dout, axis=0, keepdims=True)  # (1, output_dim)
        return dout @ W.T             # Gradient für vorherigen Layer


# Test: Forward
layer = Dense(5, 3)
x = np.random.randn(4, 5)
out = layer.forward(x)
print(f"Input Shape:  {x.shape}")
print(f"Output Shape: {out.shape}")
print(f"W Shape:      {layer.W.shape}")
print(f"b Shape:      {layer.b.shape}")

# Test: Backward
dout = np.random.randn(4, 3)
dx = layer.backward(dout)
print(f"\ndx Shape:  {dx.shape}  (muss (4,5) sein)")
print(f"dW Shape:  {layer.dW.shape}  (muss (5,3) sein)")
print(f"db Shape:  {layer.db.shape}  (muss (1,3) sein)")

---
## 3. Cross-Entropy Loss

Der Standard-Loss für Klassifikation:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log(p_{i, y_i})$$

wobei $p_{i, y_i}$ die vorhergesagte Wahrscheinlichkeit für die richtige Klasse ist.

**Trick:** Wir kombinieren Softmax + Cross-Entropy für numerische Stabilität:
$$\mathcal{L} = \frac{1}{N} \sum \left[ \log\left(\sum e^{x_j}\right) - x_{y_i} \right]$$

**Gradient (kombiniert):**
$$\frac{\partial \mathcal{L}}{\partial x_i} = \frac{1}{N}(p_i - \mathbb{1}_{i=y})$$

In [ ]:
class CrossEntropyLoss:
    """Cross-Entropy Loss mit kombiniertem Softmax."""
    def __init__(self):
        self.cache = None
        self.y_true = None

    def forward(self, logits, y_true):
        """
        Args:
            logits: (N, C) — Rohwerte vor Softmax
            y_true: (N,)   — Integer-Labels 0..C-1
        """
        self.y_true = y_true
        N = logits.shape[0]

        # Numerisch stabil: log(softmax) = logits - log(sum(exp(logits)))
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        log_sum_exp = np.log(np.sum(np.exp(shifted), axis=1))
        correct_logits = shifted[np.arange(N), y_true]

        loss = np.mean(log_sum_exp - correct_logits)
        self.cache = shifted
        return loss

    def backward(self):
        """Gradient der kombinierten Softmax + Cross-Entropy."""
        N = self.y_true.shape[0]
        shifted = self.cache
        exp = np.exp(shifted)
        probs = exp / np.sum(exp, axis=1, keepdims=True)

        # Gradient: (probs - one_hot) / N
        grad = probs.copy()
        grad[np.arange(N), self.y_true] -= 1
        return grad / N


# Test: Perfekte Vorhersage → Loss ≈ 0
loss_fn = CrossEntropyLoss()
logits = np.array([[100.0, 0.0, 0.0], [0.0, 100.0, 0.0], [0.0, 0.0, 100.0]])
y = np.array([0, 1, 2])
loss = loss_fn.forward(logits, y)
print(f"Loss (perfekt): {loss:.6f}  (erwartet: ~0)")

# Test: Gradient summiert zu 0 pro Sample
logits2 = np.random.randn(6, 4)
y2 = np.random.randint(0, 4, size=6)
loss_fn.forward(logits2, y2)
grad = loss_fn.backward()
print(f"\nGradient Shape: {grad.shape}")
print(f"Summe pro Sample: {np.sum(grad, axis=1)}  (muss [0,0,0,0,0,0] sein) ✓")

---
## 4. SGD Optimizer (mit Momentum)

**Standard SGD:**
$$W_{t+1} = W_t - \eta \cdot \frac{\partial L}{\partial W}$$

**Mit Momentum:**
$$v_{t+1} = \beta \cdot v_t - \eta \cdot \frac{\partial L}{\partial W}$$
$$W_{t+1} = W_t + v_{t+1}$$

Momentum ($\beta$) hilft:
- Schnellere Konvergenz
- Überwinden lokaler Minima
- Dämpft Oszillationen

In [ ]:
class SGD:
    """Stochastic Gradient Descent mit Momentum."""
    def __init__(self, lr=0.01, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocities = {}  # id(layer) -> {W: v_W, b: v_b}

    def step(self, layers):
        for layer in layers:
            lid = id(layer)
            if lid not in self.velocities:
                self.velocities[lid] = {"W": 0, "b": 0}

            # Momentum-Update
            self.velocities[lid]["W"] = (
                self.momentum * self.velocities[lid]["W"] - self.lr * layer.dW
            )
            self.velocities[lid]["b"] = (
                self.momentum * self.velocities[lid]["b"] - self.lr * layer.db
            )

            layer.W += self.velocities[lid]["W"]
            layer.b += self.velocities[lid]["b"]


# Test: Gewichte werden aktualisiert
layer = Dense(3, 2)
layer.W = np.ones((3, 2))
layer.b = np.ones((1, 2))
layer.dW = np.ones((3, 2)) * 0.1
layer.db = np.ones((1, 2)) * 0.1

opt = SGD(lr=0.1, momentum=0.0)
opt.step([layer])

print(f"W nach Update:\n{layer.W}")
print(f"Erwartet: 0.99 (1.0 - 0.1*0.1)")

---
## 5. Das komplette Neuronale Netzwerk

Jetzt fügen wir alles zusammen! Die `NeuralNetwork`-Klasse:
- Nimmt eine Liste von Layern (Dense + Aktivierungen)
- `forward()`: Daten durch alle Layer schicken
- `backward()`: Gradienten rückwärts durch alle Layer
- `train_step()`: Ein kompletter Trainingsschritt
- `predict()`: Vorhersage für neue Daten

In [ ]:
class NeuralNetwork:
    """
    Einfaches Feedforward-Netzwerk mit beliebig vielen Layern.

    Beispiel:
        net = NeuralNetwork([
            Dense(784, 128), ReLU(),
            Dense(128, 64),  ReLU(),
            Dense(64, 10),
        ])
    """
    def __init__(self, architecture):
        self.layers = architecture
        self.loss_fn = CrossEntropyLoss()

    def forward(self, x):
        """Forward-Pass durch alle Layer. Gibt Logits zurück."""
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, grad):
        """Backward-Pass (rückwärts durch alle Layer)."""
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def train_step(self, x, y, optimizer):
        """
        Ein Trainingsschritt: Forward → Loss → Backward → Update.
        Returns: (loss, accuracy)
        """
        # Forward
        logits = self.forward(x)

        # Loss
        loss = self.loss_fn.forward(logits, y)

        # Accuracy
        preds = np.argmax(logits, axis=1)
        acc = np.mean(preds == y)

        # Backward
        grad = self.loss_fn.backward()
        self.backward(grad)

        # Update (nur Dense-Layer haben Gewichte)
        dense_layers = [l for l in self.layers if isinstance(l, Dense)]
        optimizer.step(dense_layers)

        return loss, acc

    def predict(self, x):
        """Vorhersage: gibt Klassen-Labels zurück."""
        logits = self.forward(x)
        return np.argmax(logits, axis=1)

    def predict_proba(self, x):
        """Vorhersage: gibt Wahrscheinlichkeiten zurück."""
        logits = self.forward(x)
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp = np.exp(shifted)
        return exp / np.sum(exp, axis=1, keepdims=True)


# Test: Forward-Pass
np.random.seed(42)
x = np.random.randn(4, 5).astype(np.float32)
y = np.array([0, 1, 2, 1])

net = NeuralNetwork([Dense(5, 4), ReLU(), Dense(4, 3)])
out = net.forward(x)
print(f"Input Shape:  {x.shape}")
print(f"Output Shape: {out.shape}  (muss (4,3) sein)")

# Test: Train Step
opt = SGD(lr=0.01)
loss, acc = net.train_step(x, y, opt)
print(f"\nLoss:     {loss:.4f}")
print(f"Accuracy: {acc:.2%}")

# Test: Predict
preds = net.predict(x)
print(f"\nPredictions: {preds}")
print(f"True Labels:  {y}")

---
## 6. Training auf MNIST

Jetzt trainieren wir unser selbstgebautes Netz auf echten Daten!

**Architektur:** 784 → 128 → 64 → 10

**Daten:** MNIST (60.000 Train, 10.000 Test)

In [ ]:
import gzip, os
from urllib import request

def load_mnist():
    """Lädt MNIST aus lokalem Cache oder lädt es herunter."""
    cache_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")) or ".", ".mnist_cache")
    os.makedirs(cache_dir, exist_ok=True)

    files = {
        "train_images": "train-images-idx3-ubyte.gz",
        "train_labels": "train-labels-idx1-ubyte.gz",
        "test_images": "t10k-images-idx3-ubyte.gz",
        "test_labels": "t10k-labels-idx1-ubyte.gz",
    }

    base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"

    for fname in files.values():
        path = os.path.join(cache_dir, fname)
        if not os.path.exists(path):
            print(f"  Lade {fname} herunter...")
            request.urlretrieve(base_url + fname, path)

    def load_images(path):
        with gzip.open(path, "rb") as f:
            data = np.frombuffer(f.read(), np.uint8, offset=16)
        return data.reshape(-1, 784).astype(np.float32) / 255.0

    def load_labels(path):
        with gzip.open(path, "rb") as f:
            return np.frombuffer(f.read(), np.uint8, offset=8)

    X_train = load_images(os.path.join(cache_dir, files["train_images"]))
    y_train = load_labels(os.path.join(cache_dir, files["train_labels"]))
    X_test = load_images(os.path.join(cache_dir, files["test_images"]))
    y_test = load_labels(os.path.join(cache_dir, files["test_labels"]))

    return X_train, y_train, X_test, y_test


print("📦 Lade MNIST-Daten...")
X_train, y_train, X_test, y_test = load_mnist()
print(f"   Train: {X_train.shape[0]:,} Bilder, Test: {X_test.shape[0]:,} Bilder")
print(f"   Shape: {X_train.shape[1]} Pixel (28×28) pro Bild")

In [ ]:
# Netzwerk bauen
print("🧠 Baue Netzwerk: 784 → 128 → 64 → 10")
net = NeuralNetwork([
    Dense(784, 128), ReLU(),
    Dense(128, 64),  ReLU(),
    Dense(64, 10),
])

# Parameter zählen
total_params = sum(
    layer.W.size + layer.b.size
    for layer in net.layers if isinstance(layer, Dense)
)
print(f"   Parameter: {total_params:,}")
print(f"   Davon Gewichte: 784×128 + 128×64 + 64×10 = {784*128 + 128*64 + 64*10:,}")
print(f"   Davon Biases:   128 + 64 + 10 = {128+64+10}")

In [ ]:
# Training
print("🏋️ Training (10 Epochen)...")
optimizer = SGD(lr=0.1, momentum=0.9)
batch_size = 64
epochs = 10

for epoch in range(epochs):
    # Shuffle
    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]

    total_loss, total_acc, n_batches = 0, 0, 0

    for i in range(0, len(X_train), batch_size):
        x_batch = X_train[i : i + batch_size]
        y_batch = y_train[i : i + batch_size]

        loss, acc = net.train_step(x_batch, y_batch, optimizer)
        total_loss += loss
        total_acc += acc
        n_batches += 1

    avg_loss = total_loss / n_batches
    avg_acc = total_acc / n_batches

    # Test-Accuracy
    test_preds = net.predict(X_test)
    test_acc = np.mean(test_preds == y_test)

    print(f"   Epoche {epoch+1:2d}: "
          f"Loss={avg_loss:.4f}  "
          f"Train-Acc={avg_acc:.3f}  "
          f"Test-Acc={test_acc:.3f}")

print(f"\n✅ Training abgeschlossen!")

In [ ]:
# Finale Evaluation
test_preds = net.predict(X_test)
test_acc = np.mean(test_preds == y_test)
print(f"📊 Finale Test-Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

# Fehleranalyse
errors = [(true, pred) for true, pred in zip(y_test, test_preds) if true != pred]
print(f"   Fehlklassifikationen: {len(errors)}/{len(y_test)} ({len(errors)/len(y_test)*100:.2f}%)")

---
## 7. Gradienten-Check: Stimmt unsere Backpropagation?

Der ultimative Test: Wir vergleichen unseren analytischen Gradienten mit einer numerischen Approximation.

$$\frac{\partial f}{\partial x} \approx \frac{f(x + \epsilon) - f(x - \epsilon)}{2\epsilon}$$

In [ ]:
np.random.seed(99)
layer = Dense(3, 2)
x = np.random.randn(2, 3).astype(np.float32)

# Forward (nur für Cache)
layer.forward(x)
dout = np.random.randn(2, 2).astype(np.float32)

# Analytischer Gradient
layer.backward(dout)
dW_analytical = layer.dW.copy()

# Numerischer Gradient
eps = 1e-5
dW_numerical = np.zeros_like(layer.W)
for i in range(layer.W.shape[0]):
    for j in range(layer.W.shape[1]):
        layer.W[i, j] += eps
        out_plus = layer.forward(x)
        loss_plus = np.sum(out_plus * dout)

        layer.W[i, j] -= 2 * eps
        out_minus = layer.forward(x)
        loss_minus = np.sum(out_minus * dout)

        layer.W[i, j] += eps  # restore
        dW_numerical[i, j] = (loss_plus - loss_minus) / (2 * eps)

# Relativer Fehler
rel_error = np.max(np.abs(dW_analytical - dW_numerical) /
                   (np.abs(dW_analytical) + np.abs(dW_numerical) + 1e-8))
print(f"Relativer Fehler: {rel_error:.2e}")
if rel_error < 1e-5:
    print("✅ Gradienten-Check BESTANDEN! Backprop ist korrekt.")
else:
    print("❌ Gradienten-Check FEHLGESCHLAGEN! Backprop hat einen Fehler.")

---
## Zusammenfassung: Was haben wir gebaut?

| Komponente | Forward | Backward |
|-----------|---------|----------|
| **Dense** | $y = xW + b$ | $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} W^T$ |
| **ReLU** | $\max(0, x)$ | $\frac{\partial L}{\partial x} \cdot \mathbb{1}_{x>0}$ |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | $\frac{\partial L}{\partial x} \cdot \sigma(1-\sigma)$ |
| **CrossEntropy** | $-\frac{1}{N}\sum \log(p_{y})$ | $\frac{1}{N}(p - \text{one\_hot})$ |
| **SGD+Momentum** | $W_{t+1} = W_t + v_{t+1}$ | $v_{t+1} = \beta v_t - \eta \frac{\partial L}{\partial W}$ |

### Der Trainingsloop in 5 Schritten:

```
1. FORWARD:  Daten durch alle Layer schicken → Logits
2. LOSS:     Cross-Entropy zwischen Logits und Labels
3. BACKWARD: Gradienten rückwärts durch alle Layer (Kettenregel)
4. UPDATE:   Gewichte mit SGD+Momentum anpassen
5. REPEAT:   Nächster Batch
```

### Warum das alles selbst bauen?

- **Tiefes Verständnis**: Du weißt genau, was in jedem Schritt passiert
- **Debugging**: Wenn PyTorch seltsame Ergebnisse liefert, verstehst du warum
- **Flexibilität**: Du kannst jede Komponente anpassen oder ersetzen
- **Mathematik**: Backpropagation ist die Kettenregel — nicht mehr, nicht weniger!

**Ergebnis:** ~97% Accuracy auf MNIST — mit nur NumPy und ~100 Zeilen Code! 🎉